In [1]:
import sqlite3
import pandas as pd
from scipy.stats import mannwhitneyu

conn = sqlite3.connect('cell-count.db')

In [2]:
part2_query = """
    WITH sample_totals AS (
        SELECT sample, SUM(count) as total_count
        FROM cell_counts
        GROUP BY sample
    )
    SELECT 
        c.sample,
        s.total_count,
        c.population,
        c.count,
        ROUND((CAST(c.count AS FLOAT) / s.total_count) * 100, 2) AS percentage
    FROM cell_counts c
    JOIN sample_totals s ON c.sample = s.sample
    ORDER BY c.sample, c.population;
"""

df_frequencies = pd.read_sql(part2_query, conn)
display(df_frequencies.head(10))

,sample,total_count,population,count,percentage
0,sample00000,93214,b_cell,10908,11.70
1,sample00000,93214,cd4_t_cell,20491,21.98
2,sample00000,93214,cd8_t_cell,24440,26.22
3,sample00000,93214,monocyte,23511,25.22
4,sample00000,93214,nk_cell,13864,14.87
5,sample00001,100824,b_cell,6777,6.72
6,sample00001,100824,cd4_t_cell,33459,33.19
7,sample00001,100824,cd8_t_cell,19407,19.25
8,sample00001,100824,monocyte,23011,22.82
9,sample00001,100824,nk_cell,18170,18.02


In [3]:
part3_query = """
    WITH sample_totals AS (
        SELECT sample, SUM(count) as total_count
        FROM cell_counts
        GROUP BY sample
    )
    SELECT 
        m.sample,
        m.response,
        c.population,
        ROUND((CAST(c.count AS FLOAT) / s.total_count) * 100, 2) AS percentage
    FROM samples m
    JOIN cell_counts c ON m.sample = c.sample
    JOIN sample_totals s ON m.sample = s.sample
    WHERE m.condition = 'melanoma' 
      AND m.treatment = 'miraclib' 
      AND m.sample_type = 'PBMC';
"""

df_stats = pd.read_sql(part3_query, conn)
display(df_stats.head())

,sample,response,population,percentage
0,sample00000,no,b_cell,11.70
1,sample00000,no,cd8_t_cell,26.22
2,sample00000,no,cd4_t_cell,21.98
3,sample00000,no,nk_cell,14.87
4,sample00000,no,monocyte,25.22


In [4]:
responders = df_stats[df_stats['response'] == 'yes']
non_responders = df_stats[df_stats['response'] == 'no']

populations = df_stats['population'].unique()
results = []

for cell_pop in populations:
    res_freq = responders[responders['population'] == cell_pop]['percentage']
    non_res_freq = non_responders[non_responders['population'] == cell_pop]['percentage']
    
    stat, p_value = mannwhitneyu(res_freq, non_res_freq, alternative='two-sided')
    
    is_significant = 'Yes' if p_value < 0.05 else 'No'
    
    results.append({
        'Cell Population': cell_pop,
        'P-Value': p_value,
        'Significant Difference': is_significant
    })

df_results = pd.DataFrame(results)
display(df_results)

,Cell Population,P-Value,Significant Difference
0,b_cell,0.055748,No
1,cd8_t_cell,0.639199,No
2,cd4_t_cell,0.013407,Yes
3,nk_cell,0.121060,No
4,monocyte,0.163509,No


In [5]:
adhoc_query = """
    SELECT ROUND(AVG(c.count), 2) AS avg_b_cells
    FROM samples s
    JOIN cell_counts c ON s.sample = c.sample
    WHERE s.condition = 'melanoma'
      AND s.sex = 'M'
      AND s.response = 'yes'
      AND s.time_from_treatment_start = 0
      AND c.population = 'b_cell';
"""

df_adhoc = pd.read_sql(adhoc_query, conn)
display(df_adhoc)

,avg_b_cells
0,10206.15
